# 策略概述

**HDBSCAN** 是命題 1 的核心：以**機器學習密度聚類取代 GICS 靜態產業分類**來建立配對搜尋空間。

作法是把每檔股票用「其報酬對共同風險因子的暴露」（前 5 個主成分的因子載荷）定位，讓演算法自動把**風險暴露相近**的股票分成一群，群內再找配對。檢驗的問題是：**資料驅動的分群，能不能找到比靜態產業分類更高品質的配對？**


# 策略架構

與傳統基準**唯一的差異是「分組」層**——用 HDBSCAN 聚類取代 GICS 產業；其餘篩選、排序、交易完全相同，確保比較是乾淨的單變因對照。

```{mermaid}
flowchart LR
  P["形成期日報酬<br/>取 5 維風險因子載荷"] --> G["分組<br/>HDBSCAN 密度聚類（取代 GICS）"]
  G --> F["篩選<br/>群內共整合 + 半衰期 + Hurst"]
  F --> R["排序<br/>SSD ＋ DTW 融合"]
  R --> T["Top N 配對<br/>→ 交易期"]
```

| 層 | 本策略採用 | 用途 |
| :--- | :--- | :--- |
| 分組 | **HDBSCAN 密度聚類**（5 維報酬因子載荷） | 資料驅動分群，取代 GICS |
| 篩選 | 群內共整合 + 半衰期 + Hurst | 確認價差均值回歸 |
| 排序 | SSD 與 DTW 主成分融合 | 綜合兩種距離 |
| 交易 | Z-Score（標準化空間） | 偏離進場、回歸出場 |


## 為何取 5 維因子載荷

報酬 PCA 的主成分依解釋變異量遞減排列：前少數主成分對應市場、規模、產業景氣等**強共同因子**，
後段主成分解釋變異低、載荷值以雜訊成分為主。

密度聚類以歐氏距離估計密度——特徵向量中每一維**等權**參與距離計算。
若保留過多低訊號維度，雜訊維度的隨機差異會稀釋強因子維度上的密度結構，
使群落邊界模糊：可聚類的股票被誤判為噪音、群數不穩定。

取 $k = 5$ 使聚類座標只由訊號最強的因子構成：

- 每檔股票的 5 維向量 ≈ 它在五大共同風險因子上的暴露組合
- 因子暴露相近的股票在此空間中自然聚攏，密度群落即「暴露結構相似的股票群」
- 對每期形成窗的雜訊實現不敏感，滾動窗之間的聚類結果更穩定


# 參考文獻與引用對應


## 文獻 1：Avellaneda & Lee (2010)

> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance, 10*(7), 761–782.

**參考部分**：

- 以 PCA 對報酬相關矩陣做特徵分解萃取共同風險因子（eigenportfolios），並指出**有效因子數量遠小於股票數量**：少數主成分即涵蓋市場的主要系統性變異
- 特徵向量以 $\sqrt{\text{特徵值}}$ 尺度化

**為何參考**：

- 「只取前少數強因子作為股票的風險表徵」即本策略取 $k=5$ 的理論基礎；其餘載荷計算方式與尺度化同前



## 文獻 2：Sarmento & Horta (2020)

> Sarmento, S. M., & Horta, N. (2020). Enhancing a pairs trading strategy with the application of machine learning. *Expert Systems with Applications, 158*, 113490.

**參考部分**：

- 「**降維** → 無監督聚類 → 規則化配對篩選」三段式框架——其中降維步驟的目的即為聚類提供**低維、高訊號**的表徵

**為何參考**：

- 本策略把框架中的降維步驟落實為「PCA 取前 5 主成分」：聚類品質依賴輸入表徵的訊噪比，降維是框架的必要成分而非可選項



## 文獻 3：Campello, Moulavi & Sander (2013)

> Campello, R. J., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD 2013*.

**參考部分**：

- HDBSCAN 演算法本體：層次密度估計、自動群數決定、噪音標記（`label = -1`）
- 密度估計對輸入空間距離度量的依賴性——距離中的雜訊維度直接影響密度結構的清晰度

**為何參考**：

- 自動群數決定與離群過濾同前；其密度估計原理亦為本策略「壓縮特徵維度以保全密度訊號」的依據



## 文獻 4：許鈞翔 (2025)／Engle & Granger (1987)／Krauss, Do & Huck (2016)

> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。元智大學碩士論文。
> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction. *Econometrica, 55*(2).
> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies. *EJOR*.

**參考部分與理由**：

- 許鈞翔 (2025)：群內配對的完整篩選與排序流程（Engle-Granger $p<0.01$ → SSD/DTW → PC1 融合排序）
- Engle & Granger：雙向 OLS + ADF 兩步驟共整合程序
- Krauss et al.：半衰期（$1$–$42$ 日）與 Hurst（$H<0.5$）門檻


# 各階段行為

策略在每個滾動形成窗（252 交易日，每 21 日滾動）內依序執行以下六個階段。


## 階段 1：5 維報酬 PCA 因子載荷萃取

1. 日報酬矩陣逐股標準化（PCA 等同對報酬**相關矩陣**做特徵分解）：

$$R \in \mathbb{R}^{(T-1) \times N}, \qquad R^{s}_{\cdot,i} = \frac{R_{\cdot,i} - \mu_i}{\sigma_i}$$

2. PCA 取 $k = \min(5,\ T-2,\ N-1)$ 個主成分
3. 每檔股票的特徵向量（$\sqrt{\text{特徵值}}$ 加權）：

$$\text{loadings}_i = \text{components}_{:,i} \times \sqrt{\text{explained variance}} \in \mathbb{R}^{5}$$

聚類座標只由訊號最強的 5 個共同因子構成（取 5 維的理由見「為何取 5 維因子載荷」一節）。


## 階段 2：HDBSCAN 密度聚類

對 $N \times 5$ 的 loadings 矩陣直接執行 HDBSCAN（`reduce_method="none"`）：

| 參數 | 值 |
| :--- | :---: |
| `min_cluster_size` | 5（並以 $\max(2, N/5)$ 上限保護小樣本窗） |
| `min_samples` | 2 |
| `metric` | euclidean |

- 噪音點（`label = -1`）映射為 `"Unknown"`，不參與配對
- 全數為噪音時 `min_cluster_size` 逐步下調（至下限 2）重試


## 階段 3：群內雙向 OLS 與統計過濾

以群集標籤（`Cluster_0`, `Cluster_1`, ...）作為分組，執行：

1. **雙向 OLS + ADF 方向決定**：兩個回歸方向各檢定一次，取 p 值較小者
2. **三道統計過濾**：ADF $p < 0.01$ → OU 半衰期 $1 \le HL \le 42$ 日（$\lambda < 0$）→ Hurst $H < 0.50$


## 階段 4：SSD 與 DTW 距離計算

$$\text{SSD}_{A,B} = \sum_{t=1}^{F} \left(P'_{A,t} - P'_{B,t}\right)^2$$

$$D(i,j) = (P'_{A,i} - P'_{B,j})^2 + \min\big\{ D(i-1,j),\ D(i,j-1),\ D(i-1,j-1) \big\}, \qquad |i-j| \le 15$$


## 階段 5：PCA 融合排序與配對選取

1. SSD 與 DTW 各自標準化 → PCA 取 PC1 分數（loadings 為負則取反）
2. 依 PC1 分數升序取前 `top_n` 組

輸出欄位與群集／真實產業雙軌記錄（`Sector` = 群集標籤；`Sector_A/B` = 真實 GICS 回填，
供交易期對配對兩腳各自計算產業曝險）。


## 階段 6：交易期的參數使用方式

`ignore_ols_alpha=True`，交易期於標準化空間重建 spread：

$$P'_{i,t} = \frac{\ln P_{i,t} - \texttt{Log\_Mean}_i}{\texttt{Log\_Std}_i}, \qquad
\text{Spread}_t = P'_{A,t} - \texttt{Hedge\_Ratio} \cdot P'_{B,t}, \qquad
Z_t = \frac{\text{Spread}_t - \texttt{Spread\_Mean}}{\texttt{Spread\_Std}}$$

形成期統計量整個交易期凍結不變（無前視）。交易決策細節見 `trading/zscore_trading.ipynb`。
本策略的形成期配對另供 DRL 門檻選擇式交易端使用（見 `trading/drl_threshold_trading.ipynb`）。


# 參數總表

| 參數 | 值 | 對應階段 | 說明 |
| :--- | :---: | :--- | :--- |
| 形成窗 / 滾動步長 | 252 / 21 交易日 | 全流程輸入 | 約一年 / 一個月 |
| 每期配對數 | 網格 [1, 3, 5, 10, 20] | 排序（選取） | 依融合分數升序取前幾組 |
| 風險因子數 | 5 | 分組 | 聚類座標的維度（前 5 大共同因子） |
| 最小群大小 | 5 | 分組 | 群至少要幾檔股票才成立 |
| 聚類密度保守度 | 2 | 分組 | 值越大越保守、噪音越多 |
| 距離衡量 | SSD + DTW（主成分融合） | 排序 | 兩種距離綜合 |
| 共整合顯著水準 | 0.01 | 篩選 | 共整合檢定門檻 |
| 半衰期 / Hurst | $[1,\ 42]$ 日 / <0.5 | 篩選 | 均值回歸過濾 |
